<a href="https://colab.research.google.com/github/KhushMewada/FAI-PE-Microproject/blob/main/Honours_Microproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install streamlit pyngrok requests radon

In [ ]:
from google.colab import userdata

for key_name in ["OPENROUTER_API_KEY", "NGROK_AUTHTOKEN"]:
    try:
        key_value = userdata.get(key_name)
        print(key_name, "OK" if key_value else "EMPTY")
    except Exception:
        print(key_name, "MISSING")

OPENROUTER_API_KEY OK
NGROK_AUTHTOKEN OK


In [ ]:
%%writefile /content/app.py
import os
import ast
import requests
import streamlit as st
from radon.complexity import cc_visit
from radon.raw import analyze

st.set_page_config(page_title="Code Refactoring Assistant", layout="wide")
st.title("Code Refactoring Assistant")
st.write("Analyze Python code and improve its readability with AI.")

SAMPLE = '''def calculate_total(numbers):
    total = 0
    for number in numbers:
        if number > 0:
            total = total + number
    return total
'''

def analyze_code(code):
    result = {
        "Syntax valid": False,
        "Lines": 0,
        "Logical lines": 0,
        "Comments": 0,
        "Functions": 0,
        "Classes": 0,
        "Cyclomatic complexity": 0
    }
    tree = ast.parse(code)
    raw = analyze(code)
    blocks = cc_visit(code)

    result["Syntax valid"] = True
    result["Lines"] = raw.loc
    result["Logical lines"] = raw.lloc
    result["Comments"] = raw.comments
    result["Functions"] = sum(
        isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
        for n in ast.walk(tree)
    )
    result["Classes"] = sum(
        isinstance(n, ast.ClassDef) for n in ast.walk(tree)
    )
    result["Cyclomatic complexity"] = sum(
        b.complexity for b in blocks
    )
    return result

def refactor_code(code):
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OpenRouter API key is not configured.")

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "openrouter/free",
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "Refactor the supplied Python code for readability "
                        "and maintainability while preserving behavior. "
                        "Return only complete Python source code without "
                        "Markdown fences."
                    )
                },
                {"role": "user", "content": code}
            ],
            "temperature": 0.2
        },
        timeout=120
    )
    if not response.ok:
        raise RuntimeError(
            f"OpenRouter error {response.status_code}: "
            f"{response.text[:700]}"
        )

    output = response.json()["choices"][0]["message"]["content"].strip()
    if output.startswith("```"):
        lines = output.splitlines()
        lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        output = "\n".join(lines).strip()

    ast.parse(output)
    return output

original = st.text_area("Enter Python code", SAMPLE, height=250)

if st.button("Analyze Code"):
    try:
        st.subheader("Original Code Analysis")
        st.json(analyze_code(original))
    except Exception as exc:
        st.error(f"Analysis failed: {exc}")

if st.button("Refactor with OpenRouter", type="primary"):
    if not original.strip():
        st.warning("Enter Python code first.")
    else:
        try:
            with st.spinner("Refactoring code..."):
                improved = refactor_code(original)
            st.session_state["improved"] = improved
            st.session_state["original"] = original
        except Exception as exc:
            st.error(str(exc))

if st.session_state.get("improved"):
    improved = st.session_state["improved"]
    st.subheader("Refactored Code")
    st.code(improved, language="python")
    st.download_button(
        "Download Refactored Code",
        improved,
        file_name="refactored_code.py"
    )

    try:
        before = analyze_code(st.session_state["original"])
        after = analyze_code(improved)
        st.subheader("Before and After")
        st.table([
            {"Metric": key, "Original": value, "Refactored": after[key]}
            for key, value in before.items()
        ])
    except Exception as exc:
        st.warning(f"Could not compare metrics: {exc}")

st.caption(
    "Syntax validity does not guarantee behavioral equivalence. "
    "Review AI-generated code before using it. Submitted code is not "
    "executed by this prototype."
)

Overwriting /content/app.py


In [ ]:
import os
import sys
import time
import subprocess
from google.colab import userdata

env = os.environ.copy()
env["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

# Stop an older copy of this app if present.
subprocess.run(
    ["bash", "-lc", "pkill -f '[s]treamlit run /content/app.py' || true"],
    check=False
)
time.sleep(2)

log = open("/content/streamlit.log", "w")
process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(5)
print("Process running:", process.poll() is None)
print(open("/content/streamlit.log").read()[-2500:])

Process running: True


2026-10-26 17:36:19.537 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.80.249.253:8501




In [ ]:
import requests

try:
    response = requests.get("http://127.0.0.1:8501", timeout=10)
    print("HTTP status:", response.status_code)
    print("App is responding:", response.status_code == 200)
except Exception as exc:
    print("Connection failed:", repr(exc))
    print(open("/content/streamlit.log").read()[-4000:])

HTTP status: 200
App is responding: True


In [ ]:
from pyngrok import ngrok

# Close any previous ngrok tunnels
ngrok.kill()

# Reuse the token already saved in ngrok's configuration
tunnel = ngrok.connect(8501, "http")

print("Your Code Refactoring Assistant URL:")
print(tunnel.public_url)

Your Code Refactoring Assistant URL:
https://numerous-unsubtly-hamstring.ngrok-free.dev
